# Check outputs of MODIS_HRRR processing 
*J. Michelle Hu  
University of Utah  
August 2024*  
---

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import geopandas as gpd
import xarray as xr
# import hvplot.xarray

from s3fs import S3FileSystem, S3Map

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

import seaborn as sns
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

### Env setup

In [ ]:
from pathlib import PurePath
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem

pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

## Directory

In [ ]:
# basin = 'blue_river'
# basin = 'yampa'
basin = 'littleanimas'

workdir = '/uufs/chpc.utah.edu/common/home/skiles-group1/jmhu/model_animas_only/'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group1/jmhu/isnobal_scripts'
# basindirs = h.fn_list(workdir, f'*{basin}*')
# Select just for all variable output of time decay
basindirs = h.fn_list(workdir, f'*{basin}*isnobal_*')

# Get the WY from the directory name - assumes there is only one WY per basin
WY = int(h.fn_list(basindirs[0], '*')[0].split('/')[-1].split('wy')[-1])
print(WY)

# Figure out filenames
poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
print(poly_fn)

# SNOTEL all sites geojson fn - snotel site json
allsites_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/SNOTEL/snotel_sites_32613.json'

basindirs

## Specify a point for evaluation

In [ ]:
# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=allsites_fn)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

In [ ]:
def get_snotel(sitenum, sitename, ST, WY, epsg=32613):
    '''Use metloom to pull snotel coordinates and return as geodataframe and daily data as dict of dataframes'''
    from metloom.pointdata import SnotelPointData
    import geopandas as gpd
    from shapely.geometry import Point
    from datetime import datetime
    
    # start and end date
    start_date = datetime(WY-1, 10, 1)
    end_date = datetime(WY, 9, 30)

    snotel_dfs = dict()
    snotellats = []
    snotellons = []
    for snotelNUM, snotelNAME, snotelST in zip(sitenum, sitename, ST):
        snotel_point = SnotelPointData(f"{snotelNUM}:{snotelST}:SNTL", f"{snotelNAME}")

        meta_df = snotel_point.metadata
        lon, lat = meta_df.x, meta_df.y
        snotellats.append(lat)
        snotellons.append(lon)
        
        # set up variable list
        variables = [snotel_point.ALLOWED_VARIABLES.SNOWDEPTH]

        # request the data - use daily, the hourly data is too noisy and messes up SDD calcs
        df = snotel_point.get_daily_data(start_date, end_date, variables)
        # df = snotel_point.get_hourly_data(start_date, end_date, variables)

        # Convert to metric here
        df['SNOWDEPTH_m'] = df['SNOWDEPTH'] * 0.0254
        
        # Reset the index 
        df = df.reset_index().set_index("datetime")

        # Store in dict
        snotel_dfs[snotelNAME] = df
    
    # Create a Geoseries based off of a list of a Shapely point using the lat and lon from the SNOTEL site
    s = gpd.GeoSeries([Point(lon, lat) for lon, lat in zip(snotellons, snotellats)])

    # Turn this into a geodataframe and specify the geom as the geoseries of the SNOTEL point
    gdf = gpd.GeoDataFrame(geometry=s)

    # Set the CRS inplace
    gdf.set_crs('epsg:4326', inplace=True)

    # Convert snotel coords' lat lon to UTM
    gdf = gdf.to_crs(f'epsg:{epsg}')

    return gdf, snotel_dfs

In [ ]:
ST_arr = ['CO'] * len(sitenums)
gdf_metloom, snotel_dfs = get_snotel(sitenums, sitenames, ST_arr, WY=WY)
gdf_metloom

## Take a look at snow depth at these snotel points

In [ ]:
labels = ['iSnobal-HRRR', 'HRRR-SC', 'HRRR-MODIS', 'HRRR-MODIS non-negative enforcement']

In [ ]:
figsize = (18, 4)
linestyles = ['-', '--']
linewidth = 1
marker = None
color = 'k'
snotelcolors = ['dodgerblue', 'gray']
isnobalcolors = ['k', 'coral']

### Plot snow depth for all model runs

In [ ]:
%%time
# currently for one WY (only one per basin as of 20240618), will need to add more for WYs in the future
zdx = 0
month = 'run20'
varname = 'snow'
days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}.nc")
    days[label] = basin_days
    
days.keys()

In [ ]:
%%time
# extract the snow state variables for the selected sites
ds_dict = dict()
snow_var_list = ['thickness', 'temp_surf', 'temp_lower', 
                 # 'temp_snowcover', 
                 'thickness_lower']
for label in days.keys():
    ds_list = [xr.open_dataset(day_fn) for day_fn in days[label]]
    for thisvar in snow_var_list:
        snow_var_data = [ds[thisvar].sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
        snow_var_data = xr.concat(snow_var_data, dim='time')
        ds_dict[f'{label}_{thisvar}'] = snow_var_data

In [ ]:
%%time
thisvar = 'thickness'
# colors = ['navy', 'firebrick', 'dodgerblue']
colors = ['xkcd:coral', 'xkcd:easter purple', 'xkcd:bright blue', 'navy']
for kdx, sitename in enumerate(sitenames):
    fig, ax = plt.subplots(1, figsize=(16, 4))
    for mdx, label in enumerate(days.keys()):
        snow_var_data = ds_dict[f'{label}_{thisvar}']
        snow_var_data[:, kdx, kdx].plot(ax=ax, color=colors[mdx], label=label, linewidth=2, linestyle='-', alpha=0.8)
        
    # Plot WY time series of snow depth
    (snotel_dfs[f'{sitename}']['SNOWDEPTH_m']).plot(ax=ax, 
                                                    label=f'{sitename} Snow Depth [m]', 
                                                    linestyle='-', 
                                                    linewidth=1, 
                                                    color='dimgray',
                                                    marker='.',
                                                    markersize=5
                                                   )

    plt.legend(bbox_to_anchor=(1,1))
    plt.title('Snow Depth')
    plt.xlabel('');

In [ ]:
linestyles = ['-', '-', '-', ':']
linewidths = [1.5, 1.5, 1.5, 2]

## Plot shortwave!
DSWRF vs. SMRF bits

In [ ]:
%%time
# extract solar radiation components for selected sites
chunks = 'auto'
drop_var_list = ['projection']

# Loop through basindirs
sw_days = dict()
for jdx, (label, basindir) in enumerate(zip(labels, basindirs)):
    if jdx == 0:
        basin_days = []
        # For time decay, you will need to construct solar irradiance, pluck out the individual veg_rad files first for each day
        for varname in ['veg_ir_beam', 'veg_ir_diffuse', 'veg_vis_beam', 'veg_vis_diffuse']:
            veg_basin_days = h.fn_list(basindir, f"*/*/*/*{varname}*.nc")
            basin_days.append(veg_basin_days)
    else:
        # for HRRR, you can just pull DSWRF
        basin_days = h.fn_list(basindir, f"*/*/{month}*/net_solar.nc")
    sw_days[label] = basin_days
    
sw_days.keys()

In [ ]:
def calc_irrad(veg_fns, drop_var_list=['projection'], chunks='auto'):
    '''Sum total veg-corrected solar irradiance from time decay run
    solar_irrad = veg_ir_beam + veg_ir_diffuse + veg_vis_beam + veg_vis_diffuse
    '''
    veg_dslist = [xr.open_dataset(veg_fn, chunks=chunks, drop_variables=drop_var_list) for veg_fn in veg_fns ]
    for jdx, ds in enumerate(veg_dslist):
        thisvar = list(ds.variables.mapping.keys())[3]
        # print(thisvar)
        da = ds[thisvar]
        if jdx == 0:
            solar_irrad = da
        else:
            solar_irrad = solar_irrad + da
    return solar_irrad

In [ ]:
%%time
sw_dict = dict()
for jdx, label in enumerate(sw_days.keys()):
    if jdx == 0:
        ds_list = []
        for kdx, _ in enumerate(sw_days[label][0]):
            # print(kdx)
            veg_fns = [sw_days[label][0][kdx], sw_days[label][1][kdx], sw_days[label][2][kdx], sw_days[label][3][kdx]]
            solar_irrad_ds = calc_irrad(veg_fns=veg_fns)
            ds_list.append(solar_irrad_ds)
    else:
        ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in sw_days[label]]
    var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    var_data = xr.concat(var_data, dim='time')
    sw_dict[label] = var_data.resample(time='1D').mean().load()

In [ ]:
sw_dict.keys()

In [ ]:
linewidths = [0, 1.5, 1.5, 3]
linestyles = ['-', '-', '-', ':']
markers = ['o', '|', '', '']
markersizes = [5, 15, 0, 0]
alphas = [1, 1, 1, 1]

In [ ]:
%%time
# Plot them
for ldx, f in enumerate(snotel_dfs.keys()):   
    fig, ax = plt.subplots(1, figsize=(18, 4))
    # loop through each model approach
    for mdx, label in enumerate(sw_dict.keys()):
        # extract the variable
        if mdx == 0:
            da = sw_dict[label]
        else:
            da = sw_dict[label]['DSWRF']
        da[:, ldx, ldx].plot(ax=ax, 
                             color=colors[mdx], 
                             label=f'{label}', 
                             linewidth=linewidths[mdx], linestyle=linestyles[mdx], 
                             alpha=alphas[mdx],
                             marker=markers[mdx], markersize=markersizes[mdx]
                            ) 
    ax.set_xlabel('')
    ax.set_ylabel('Radiation [W/m^2]')
    ax.set_title(f'{f}: downwelling shortwave')
    ax.set_xlim([sw_dict[f'{label}'].time.values[0], sw_dict[f'{label}'].time.values[-75]])
    ax.set_ylim(0, 500)
    l = ax.legend()
        # snotelsite = '_'.join(f.split(' '))
        # plt.savefig(f'./pngs/isnobal_inputs/{label}_{varname}_{snotelsite}.png', dpi=300)

In [ ]:
# chunks = 'auto'
# test_fns = h.fn_list(basindirs[0], f"*/*/*20201030*/*.nc")
# test_fns = [fn for fn in test_fns if 'veg_' in fn]
# ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=['projection']) for day_fn in test_fns]
# fig, axa = plt.subplots(2, 6, figsize=(24, 8), sharex=True, sharey=True)

# for ax, ds in zip(axa.flatten(), ds_list):
#     thisvar = list(ds.variables.mapping.keys())[3]
#     h.plot_one(ds[thisvar][18, :, :], specify_ax=(fig, ax), turnoffaxes=True, turnofflabels=True, title=ds[thisvar].name)
    
    
# fig, axa = plt.subplots(2, 6, figsize=(18, 6), sharex=True, sharey=True)
# for ax, ds in zip(axa.flatten(), ds_list):
#     thisvar = list(ds.variables.mapping.keys())[3]
#     _ = ds[thisvar].sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest')[:, 0, 0].plot(ax=ax)
#     # ax.set_xticks(ticks=ax.get_xticks(), rotation=90)    
#     ax.set_xticklabels(labels=ax.get_xticklabels(), rotation=90)
#     ax.set_title('');
# plt.tight_layout();

# # calculate total veg-corrected irradiance
# # solar_irrad = veg_ir_beam + veg_ir_diffuse + veg_vis_beam + veg_vis_diffuse
# for jdx, ds in enumerate(ds_list[-4:]):
#     thisvar = list(ds.variables.mapping.keys())[3]
#     print(thisvar)
#     da = ds[thisvar]
#     if jdx == 0:
#         solar_irrad = da
#     else:
#         solar_irrad = solar_irrad + da
# solar_irrad[0, :, :].plot.imshow()

# # read in the dswrf and plot this
# chunks = 'auto'
# dswrf_fns = h.fn_list(basindirs[1], f"*/*/*20201030*/net_solar.nc")
# ds = xr.open_dataset(dswrf_fns[0], chunks=chunks, drop_variables=['projection'])
# # ds['DSWRF'].sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest')[:, 0, 0].plot()
# ds['DSWRF'][0, :, :].plot.imshow()

# ds['DSWRF'].sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest')[:, 0, 0].plot()
# solar_irrad.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest')[:, 0, 0].plot()

## Plot and compare all the model inputs (after SMRF has distributed)
- thermal - done (thermal.nc)
- solar - done (net_solar.nc)
- precipitation mass (precip.nc)
- soil temperature (NA)
- air temperaturen (air_temp.nc)
- vapor pressure (vapor_pressure.nc)
- wind speed (wind_speed.nc)
all except for soil temperature are in smrf_YYYYMMDD.nc
> *soil temperature is a constant value of -2.5˚C*

In [ ]:
%%time
# extract model input components for selected sites
varname = 'smrf' # smrf_20201001.nc
chunks = 'auto'
drop_var_list = ['percent_snow', 'precip_temp', 'snow_density', 'storm_days']

smrf_days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}*.nc")
    # remove energy balance bits if they exist
    # smrf_energy_balance_20201001.nc
    basin_days = [day for day in basin_days if 'energy_balance' not in day]
    smrf_days[label] = basin_days

In [ ]:
smrf_dict = dict()
for label in smrf_days.keys():
    ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in smrf_days[label]]
    var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    var_data = xr.concat(var_data, dim='time')
    smrf_dict[label] = var_data.resample(time='1D').mean()

In [ ]:
%%time
# Go through each of the components and build dicts for them by model run type
var_ts_dict = dict()

for label in smrf_days.keys():
    var_data = smrf_dict[label]
    for jdx, thisvar in enumerate(var_data.data_vars):
        print(f'{label}_{thisvar}')
        var_ts_dict[f'{label}_{thisvar}'] = var_data[thisvar].load()

In [ ]:
%%time
for ldx, f in enumerate(snotel_dfs.keys()):   
    # loop through each input
    for kdx, varname in enumerate(var_data.data_vars):
        fig, ax = plt.subplots(1, figsize=(18, 4))
        # loop through each model approach
        for mdx, label in enumerate(smrf_days.keys()):
            # extract the variable
            var_ts_dict[f'{label}_{varname}'][:, ldx, ldx].plot(ax=ax, 
                                                                color=colors[mdx], 
                                                                label=f'{label}', 
                                                                linewidth=linewidths[mdx], linestyle=linestyles[mdx], 
                                                                alpha=alphas[mdx],
                                                                marker=markers[mdx], markersize=markersizes[mdx]
                                                               ) 
        ax.set_xlabel('')
        # ax.set_ylabel('Radiation [W/m^2]')
        ax.set_title(f'{f}: {varname}')
        ax.set_xlim([var_ts_dict[f'{label}_{varname}'].time.values[0], var_ts_dict[f'{label}_{varname}'].time.values[-75]])
        # ax.set_ylim(ylims)
        l = ax.legend()
        snotelsite = '_'.join(f.split(' '))
        # plt.savefig(f'./pngs/isnobal_inputs/{label}_{varname}_{snotelsite}.png', dpi=300)

## Check out modified radiation components (net solar, DSWRF, albedo)

In [ ]:
%%time
# extract the net solar for the selected sites
varname = 'net_solar'
chunks = 'auto'
drop_var_list = ['projection']

solar_days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}.nc")
    solar_days[label] = basin_days

solar_dict = dict()
for label in solar_days.keys():
    ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in solar_days[label]]
    snow_var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    snow_var_data = xr.concat(snow_var_data, dim='time')
    solar_dict[label] = snow_var_data.resample(time='1D').mean()

## Plot just the net solar

In [ ]:
%%time
varname = 'net_solar'
for kdx, f in enumerate(snotel_dfs.keys()):
    fig, ax = plt.subplots(1, figsize=(18, 4))
    ax.axhline(y=0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], color='dimgray', linestyle=':')
    
    for ldx, label in enumerate(solar_days.keys()):
        print(label)
        var_data = solar_dict[label]
        var_data[varname][:, kdx, kdx].plot(ax=ax, label=label, linewidth=linewidths[ldx], linestyle=linestyles[ldx], color=colors[ldx]) 

    ax.set_ylim(-100, 500)
    ax.set_ylabel('Radiation [W/m^2]')
    ax.legend(ncol=(len(solar_days.keys())))
    ax.set_title('{f}: net solar')

# Look at energy balance components (em.nc) outputs below

In [ ]:
%%time
# extract radiation components for selected sites
varname = 'em'
chunks = 'auto'
drop_var_list = ['evaporation', 'snowmelt', 'SWI', 'projection']

em_days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}.nc")
    em_days[label] = basin_days

em_dict = dict()
for label in em_days.keys():
    ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in em_days[label]]
    snow_var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    snow_var_data = xr.concat(snow_var_data, dim='time')
    em_dict[label] = snow_var_data.resample(time='1D').mean()

In [ ]:
%%time
# Go through each of the energy balance components and build dicts for them by model run type
var_ts_dict = dict()
updated_var_dict = dict()

for label in em_days.keys():
    var_data = em_dict[label]
    for jdx, thisvar in enumerate(var_data.data_vars):
        print(f'{label}_{thisvar}')
        var_ts_dict[f'{label}_{thisvar}'] = var_data[thisvar].load()

In [ ]:
# fig, ax = plt.subplots(figsize=(18, 4))
# for label in em_days.keys():
#     snow_var_data = em_dict[label]
#     snow_var_data['net_rad'][:, 0, 0].plot(ax=ax, label=label)
#     ax.set_title('')
# plt.title('net_rad')    
# plt.legend()    

### Compute net longwave

In [ ]:
# %%time
# # Double check net longwave by computing from subtracting net solar from net radiation 
# # linestyles = ['-', '-', ':']
# # linewidths = [1.5, 1.5, 2]
# ylims = (-500, 500)

# # Loop through snotel sites
# for ldx, f in enumerate(snotel_dfs.keys()):   
#     # loop through each energy balance component
#     for kdx, varname in enumerate(var_data.data_vars):
#         fig, ax = plt.subplots(1, figsize=(18, 4))
#         ax.axhline(y=0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], color='dimgray', linestyle=':')
#         # loop through each model approach
#         for mdx, label in enumerate(em_days.keys()):
#             # extract the em.nc variable - first is net radiation            
#             netsolar_data = solar_dict[label]
#             net_rad_data = var_ts_dict[f'{label}_{varname}']
#             net_longwave = net_rad_data[:, ldx, ldx] - netsolar_data['net_solar'][:, ldx, ldx]
#             net_longwave.plot(ax=ax, label=f'{label}', linewidth=linewidths[mdx], linestyle=linestyles[mdx], color=colors[mdx]) 

#         ax.set_xlabel('')
#         ax.set_ylabel('Radiation [W/m^2]')
#         ax.set_title(f'{f}: Net longwave')
#         ax.set_xlim([var_ts_dict[f'{label}_{varname}'].time.values[0], var_ts_dict[f'{label}_{varname}'].time.values[-60]])
#         ax.set_ylim(ylims)
#         l = ax.legend()
#         break
#     break

## Plot the components one by one 

In [ ]:
%%time

# colors = ['xkcd:coral', 'xkcd:aqua green', 'xkcd:warm grey', 'xkcd:pastel purple', 'xkcd:bright blue', 'k']
# colors = ['xkcd:coral', 'xkcd:bright blue', 'xkcd:warm grey', 'k']
# linewidths = [1.5, 1, 1, 1, 1, 1.25]
# linestyles = ['-', '-', '-', '-', '-', '--']
# ylims = (-100, 100)


# linestyles = ['-', '-', ':']
# linewidths = [1.5, 1.5, 2]
ylims = (-75, 75)

# Plot the bits
# Loop through snotel sites
for ldx, f in enumerate(snotel_dfs.keys()):   
    # loop through each energy balance component
    for kdx, varname in enumerate(var_data.data_vars):
        fig, ax = plt.subplots(1, figsize=(18, 4))
        # loop through each model approach
        for mdx, label in enumerate(em_days.keys()):
            # extract the variable
            var_ts_dict[f'{label}_{varname}'][:, ldx, ldx].plot(ax=ax, 
                                                                color=colors[mdx], 
                                                                label=f'{label}', 
                                                                linewidth=linewidths[mdx], linestyle=linestyles[mdx]) 
        ax.set_xlabel('')
        ax.set_ylabel('Radiation [W/m^2]')
        ax.set_title(f'{f}: {varname}')
        ax.set_xlim([var_ts_dict[f'{label}_{varname}'].time.values[0], var_ts_dict[f'{label}_{varname}'].time.values[-75]])
        ax.set_ylim(ylims)
        l = ax.legend()
        # for idx, text in enumerate(l.get_texts()):
        #     text.set_color(colors[idx])

In [ ]:
for ldx, f in enumerate(snotel_dfs.keys()):   
    # loop through each energy balance component
    fig, ax = plt.subplots(1, figsize=(18, 4))
    # loop through each model approach
    for mdx, label in enumerate(em_days.keys()):
        # extract the variable
        (var_ts_dict[f'{label}_{varname}'][:, ldx, ldx]/1e6).plot(ax=ax, 
                                                            color=colors[mdx], 
                                                            label=f'{label}', 
                                                            linewidth=linewidths[mdx], linestyle=linestyles[mdx]) 
    ax.set_xlabel('')
    ax.set_ylabel('Energy [MJ/m^2]')
    ax.set_title(f'{f}: {varname}')
    ax.set_xlim([var_ts_dict[f'{label}_{varname}'].time.values[0], var_ts_dict[f'{label}_{varname}'].time.values[-75]])
    # ax.set_ylim(ylims)
    l = ax.legend()

## thermal.nc

In [ ]:
%%time
# extract the thermal for the selected sites
varname = 'thermal'
chunks = 'auto'
drop_var_list = ['projection']

solar_days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}.nc")
    if len(basin_days) == 0:
        # handling for smrf-compacted bits
        basin_days = h.fn_list(basindir, f"*/*/{month}*/combined/{varname}.nc")
    solar_days[label] = basin_days

In [ ]:
solar_dict = dict()
for label in solar_days.keys():
    print(label)
    ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in solar_days[label]]
    snow_var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    snow_var_data = xr.concat(snow_var_data, dim='time')
    solar_dict[label] = snow_var_data.resample(time='1D').mean()

In [ ]:
linewidths = [4, 1.5, 1.5, 2]

In [ ]:
%%time
for kdx, f in enumerate(snotel_dfs.keys()):
    fig, ax = plt.subplots(1, figsize=(18, 4))
    ax.axhline(y=0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], color='dimgray', linestyle=':')
    
    for ldx, label in enumerate(solar_days.keys()):
        print(label)
        var_data = solar_dict[label]
        var_data[varname][:, kdx, kdx].plot(ax=ax, label=label, linewidth=linewidths[ldx], linestyle=linestyles[ldx], color=colors[ldx]) 

    # ax.set_ylim(-100, 500)
    ax.set_ylabel('Radiation [W/m^2]')
    ax.legend(ncol=(len(solar_days.keys())))
    ax.set_title('Thermal')

## snow.nc get the bits - grabbing from above

In [ ]:
%%time
for thisvar in ['temp_surf', 'temp_lower', 'thickness', 'thickness_lower']:
    for kdx, sitename in enumerate(sitenames):
        fig, ax = plt.subplots(1, figsize=(16, 4))
        for mdx, label in enumerate(days.keys()):
            snow_var_data = ds_dict[f'{label}_{thisvar}']
            snow_var_data[:, kdx, kdx].plot(ax=ax, color=colors[mdx], label=label, linewidth=2, linestyle='-', alpha=0.6)
        plt.legend(bbox_to_anchor=(1,1))
        plt.title(f'{sitename} {thisvar}')
        plt.xlabel('');

## Downwelling Shortwave (incoming SW)
HRRR-shortwave (DSWRF) vs. the SMRF (topocalc) two-stream re-do this and write out all outputs

## Cloud cover, cloud fraction
higher cloud cover in HRRR than back-calculated time decay --> more cloud cover = more LW, slightly less SW  
lower cloud cover in HRRR --> less LW, more SW  

If it's cloud cover, I'd expect way more cloud cover in HRRR than time decay

In [ ]:
%%time
# extract the cloud_factor for the selected sites
varname = 'cloud_factor'
chunks = 'auto'
drop_var_list = ['projection']

cloud_days = dict()
for label, basindir in zip(labels, basindirs):
    basin_days = h.fn_list(basindir, f"*/*/{month}*/{varname}.nc")
    if len(basin_days) == 0:
        # handling for smrf-compacted bits
        basin_days = h.fn_list(basindir, f"*/*/{month}*/combined/{varname}.nc")
    cloud_days[label] = basin_days

In [ ]:
cloud_dict = dict()
for label in cloud_days.keys():
    print(label)
    ds_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in cloud_days[label]]
    var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in ds_list]
    var_data = xr.concat(var_data, dim='time')
    # cloud_dict[label] = var_data.resample(time='1D').mean()
    cloud_dict[label] = var_data

In [ ]:
# %%time
# for kdx, sitename in enumerate(sitenames):
#     fig, ax = plt.subplots(1, figsize=(16, 4))
#     for mdx, label in enumerate(cloud_days.keys()):
#         var_data = cloud_dict[f'{label}']
#         # add flexible handling for cloud_factor and TCDC variable names
#         thisvar = list(var_data.variables.mapping.keys())[0]
#         var_data[thisvar][:, kdx, kdx].plot(ax=ax, color=colors[mdx], label=label, linewidth=2, linestyle='-', alpha=0.6)
#     plt.legend(bbox_to_anchor=(1,1))
#     plt.title(f'{sitename} cloud factor')
#     plt.xlabel('');

In [ ]:
%%time
for kdx, sitename in enumerate(sitenames):
    for mdx, label in enumerate(cloud_days.keys()):
        var_data = cloud_dict[f'{label}']
        # add flexible handling for cloud_factor and TCDC variable names
        thisvar = list(var_data.variables.mapping.keys())[0]
        if mdx == 0:
            timedecay = var_data[thisvar][:, kdx, kdx]
        else:
            fig, ax = plt.subplots(1, figsize=(16, 4))
            (var_data[thisvar][:, kdx, kdx] - timedecay).plot(ax=ax, color=colors[mdx], label=label, linewidth=1, linestyle='-', alpha=0.6)
            # ax.legend(bbox_to_anchor=(1,1))
            ax.set_title(f'{sitename} cloud factor {label} difference from time decay')
            ax.set_xlabel('');
            ax.annotate(text=f'More clouds in {label}', xy=(0.0, 0.65), xycoords='axes fraction')
            ax.annotate(text='More clouds in time decay', xy=(0.0, 0.35), xycoords='axes fraction')
            ax.set_ylim(-1, 1)
            ax.axhline(0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], linestyle='--', linewidth=1, color='k')

In [ ]:
%%time
for kdx, sitename in enumerate(sitenames):
    for mdx, label in enumerate(cloud_days.keys()):
        var_data = cloud_dict[f'{label}']
        # add flexible handling for cloud_factor and TCDC variable names
        thisvar = list(var_data.variables.mapping.keys())[0]
        if mdx == 0:
            timedecay = var_data[thisvar][:, kdx, kdx]
        else:
            fig, ax = plt.subplots(1, figsize=(16, 4))
            (var_data[thisvar][:, kdx, kdx] - timedecay).resample(time='1D').mean().plot(ax=ax, color=colors[mdx], label=label, linewidth=1, linestyle='-', alpha=0.8)
            # ax.legend(bbox_to_anchor=(1,1))
            ax.set_title(f'{sitename} cloud factor {label} difference from time decay')
            ax.set_xlabel('');
            ax.annotate(text=f'More clouds in {label}', xy=(0.0, 0.65), xycoords='axes fraction')
            ax.annotate(text='More clouds in time decay', xy=(0.0, 0.35), xycoords='axes fraction')
            ax.set_ylim(-1, 1)
            ax.axhline(0, xmin=ax.get_xlim()[0], xmax=ax.get_xlim()[1], linestyle='--', linewidth=1, color='k')

In [ ]:
%%time
# Double check that all of the HRRR bits are the same too
for kdx, sitename in enumerate(sitenames):
    for mdx, label in enumerate(cloud_days.keys()):
        var_data = cloud_dict[f'{label}']
        # add flexible handling for cloud_factor and TCDC variable names
        thisvar = list(var_data.variables.mapping.keys())[0]
        if mdx == 0:
            pass
        elif mdx == 1:
            hrrr_sc = var_data[thisvar][:, kdx, kdx]
        else:
            fig, ax = plt.subplots(1, figsize=(16, 4))
            (var_data[thisvar][:, kdx, kdx] - hrrr_sc).plot(ax=ax, color=colors[mdx], label=label, linewidth=1, linestyle='-', alpha=0.8)
            # ax.legend(bbox_to_anchor=(1,1))
            ax.set_title(f'{sitename} cloud factor {label} difference from HRRR-SC cloud factor')
            ax.set_xlabel('');
            ax.annotate(text=f'More clouds in {label}', xy=(0.0, 0.65), xycoords='axes fraction')
            ax.annotate(text='More clouds in HRRR-SC', xy=(0.0, 0.35), xycoords='axes fraction')

# Archived

In [ ]:
figsize = (18, 4)
linestyles = ['-', '--']
linewidth = 1
marker = None
color = 'k'
snotelcolors = ['dodgerblue', 'gray']
isnobalcolors = ['k', 'coral']

In [ ]:
%%time
# Plot the bits
albedocolor = 'g'

for kdx, f in enumerate(snotel_dfs.keys()):
    fig, ax = plt.subplots(1, figsize=(18, 4))
    var_data[varname][:, kdx, kdx].plot(ax=ax, 
                #   linestyle=linestyles[0], 
                  color=isnobalcolors[0], label=labels[zdx], linewidth=1) 
    updated_var_data[varname][:, kdx, kdx].plot(ax=ax, linestyle=linestyles[1], color=isnobalcolors[1], label=labels[zdx+1], linewidth=0.6)
    updated_var_data['DSWRF'][:, kdx, kdx].plot(ax=ax, linestyle=linestyles[1], color='gray', label='DSWRF', linewidth=0.6, alpha=0.5)
    ax.set_ylim(0, 1500)
    ax.set_ylabel('Radiation [W/m^2]')

    # Plot albedo as well!
    ax2 = ax.twinx()
    updated_var_data['albedo'][:, kdx, kdx].plot(ax=ax2, color=albedocolor, label=f'{labels[zdx+1]} albedo', linewidth=2)
    ax2.set_ylim(0, 10000)
    ax2.set_ylabel('Albedo * 10000', color=albedocolor)
    
    ax2.yaxis.label.set_color(albedocolor)          #setting up Y-axis label color to blue
    ax2.tick_params(axis='y', colors=albedocolor)  #setting up Y-axis tick color to black

    ax.legend(bbox_to_anchor=(0.15, 1))
    ax2.legend(bbox_to_anchor=(1,1))
    
    ax.set_title('')
    ax2.set_title(f);

## Difference the net solars!

In [ ]:
%%time
# Plot the bits
for kdx, f in enumerate(snotel_dfs.keys()):
    diff = updated_var_data[varname][:, kdx, kdx] - var_data[varname][:, kdx, kdx]
    fig, ax = plt.subplots(1, figsize=(18, 4))
    diff.plot(ax=ax, color=isnobalcolors[0], label='Updated - original', linewidth=1) 
    ax.set_ylim(-50, 50)
    ax.set_ylabel('Difference in Radiation [W/m^2]')
    ax.legend()
    ax.set_title('')

## From the shift in  net solar, we should see the impact in energy balance terms and overall radiation balance

In [ ]:
%%time
# currently for one WY (only one per basin as of 20240618), will need to add more for WYs in the future
month = 'run20'
varname = 'em'
days = h.fn_list(basindirs[zdx], f"*/*/{month}*/{varname}.nc")
days = [days, h.fn_list(basindirs[zdx+1], f"*/*/{month}*/{varname}.nc")]

print(len(days[0]), len(days[1]))

chunks = 'auto'
drop_var_list = ['evaporation', 'snowmelt', 'SWI', 'cold_content', 'projection']

isnobal_hrrr_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in days[0]]
modis_hrrr_list = [xr.open_dataset(day_fn, chunks=chunks, drop_variables=drop_var_list) for day_fn in days[1]]

var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in isnobal_hrrr_list]
updated_var_data = [ds.sel(x=list(gdf_metloom.geometry.x.values), y=list(gdf_metloom.geometry.y.values), method='nearest') for ds in modis_hrrr_list]

# Concatenate all the days
var_data = xr.concat(var_data, dim='time')
updated_var_data = xr.concat(updated_var_data, dim='time')

In [ ]:
%%time
var_ts_dict = dict()
updated_var_dict = dict()

for jdx, thisvar in enumerate(var_data.data_vars):
    var_ts_dict[thisvar] = var_data[thisvar].load()
    updated_var_dict[thisvar] = updated_var_data[thisvar].load()

### Move the legend so it is across the top or bottom

In [ ]:
%%time

colors = ['xkcd:coral', 'xkcd:aqua green', 'xkcd:warm grey', 'xkcd:pastel purple', 'xkcd:bright blue', 'k']
linewidths = [1.5, 1, 1, 1, 1, 1.25]
linestyles = ['-', '-', '-', '-', '-', '--']
ylims = (-100, 100)

# Plot the bits
for ldx, f in enumerate(snotel_dfs.keys()):
    print(f)
    fig, ax = plt.subplots(1, figsize=(18, 4))
    for kdx, varname in enumerate(var_ts_dict.keys()):
        var_ts_dict[varname][:, ldx, ldx].plot(ax=ax, color=colors[kdx], label=f'{varname}', 
                                               linewidth=linewidths[kdx], linestyle=linestyles[kdx]) 
    ax.set_xlabel('')
    ax.set_ylabel('Radiation [W/m^2]')
    ax.set_title(f'{f}: {labels[zdx]}')
    ax.set_xlim([var_ts_dict[thisvar].time.values[0], var_ts_dict[thisvar].time.values[-75]])
    ax.set_ylim(ylims)
    l = ax.legend()
    for idx, text in enumerate(l.get_texts()):
        text.set_color(colors[idx])
    
    fig, ax = plt.subplots(1, figsize=(18, 4))
    for kdx, varname in enumerate(updated_var_dict.keys()):
        updated_var_dict[varname][:, ldx, ldx].plot(ax=ax, color=colors[kdx], label=f'{varname}', 
                                               linewidth=linewidths[kdx], linestyle=linestyles[kdx]) 
    ax.set_xlabel('')
    ax.set_ylabel('Radiation [W/m^2]')
    ax.set_title(f'{f}: {labels[zdx+1]}')
    ax.set_xlim([updated_var_dict[thisvar].time.values[0], updated_var_dict[thisvar].time.values[-75]])
    ax.set_ylim(ylims)
    l = ax.legend()
    for idx, text in enumerate(l.get_texts()):
        text.set_color(colors[idx])

## Plot a handful of days of net solar for the entire domain

## Plots some diffs

In [ ]:
%%time

colors = ['xkcd:coral', 'xkcd:aqua green', 'xkcd:warm grey', 'xkcd:pastel purple', 'xkcd:bright blue', 'k']
linewidths = [1.5, 1, 1, 1, 1, 1.25]
linestyles = ['-', '-', '-', '-', '-', '--']
ylims = (-100, 100)

# Plot the bits
for jdx, f in enumerate(snotel_dfs.keys()):
    print(f)
    fig, ax = plt.subplots(1, figsize=(18, 4))
    for kdx, varname in enumerate(var_ts_dict.keys()):
        # MODIS-HRRR difference from iSnobal HRRR (M-H minus iH)
        diff = updated_var_dict[varname][:, jdx, jdx] - var_ts_dict[varname][:, jdx, jdx]
        diff.plot(ax=ax, color=colors[kdx], label=f'{varname}', 
                                               linewidth=linewidths[kdx], linestyle=linestyles[kdx]) 
    ax.set_xlabel('')
    ax.set_ylabel('Radiation [W/m^2]')
    ax.set_title(f'{f}: MODIS-HRRR minus iSnobal-HRRR')
    ax.set_xlim([var_ts_dict[thisvar].time.values[0], var_ts_dict[thisvar].time.values[-75]])
    ax.set_ylim(ylims)
    l = ax.legend(loc='lower left')
    for idx, text in enumerate(l.get_texts()):
        text.set_color(colors[idx])

## Check out cloud factor

In [ ]:
# currently for one WY (only one per basin as of 20240618), will need to add more for WYs in the future
days = h.fn_list(basindirs[0], f"*/*/{month}*/cloud_factor.nc")
days = [days, h.fn_list(basindirs[1], f"*/*/{month}*/cloud_factor.nc")]
len(days)

In [ ]:
%%time
cloud_list = [xr.open_dataset(day_fn) for day_fn in days[0]]
tcdc_list = [xr.open_dataset(day_fn) for day_fn in days[1]]

In [ ]:
%%time
thisvar = 'Cloud Factor'
cloud_var_data = [ds['cloud_factor'].sel(x=list(gdf.geometry.x.values), y=list(gdf.geometry.y.values), method='nearest') for ds in cloud_list]
cloud_updated_var_data = [ds['TCDC'].sel(x=list(gdf.geometry.x.values), y=list(gdf.geometry.y.values), method='nearest') for ds in tcdc_list]

# Concatenate all the days
cloud_var_data = xr.concat(cloud_var_data, dim='time')
cloud_updated_var_data = xr.concat(cloud_updated_var_data, dim='time')

In [ ]:
fig, ax = plt.subplots(1, figsize=figsize)
cloud_var_data.plot(ax=ax, 
            #   linestyle=linestyles[0], 
              color='gray', label='iSnobal-HRRR', linewidth=1) 
cloud_updated_var_data.plot(ax=ax, linestyle=linestyles[1], color=isnobalcolors[1], label='MODIS-HRRR', linewidth=0.6, alpha=0.6
                      )
plt.legend()
plt.title(thisvar)

(cloud_updated_var_data - cloud_var_data).plot(figsize=figsize, color='seagreen', linewidth=0.5)
plt.title(f'{thisvar} MODIS-HRRR - iSnobal-HRRR');

## Check state values now

## Snow depth

In [ ]:
# Get disappearance dates
import processing as proc

snow_name = 'Snow Depth'
verbose = False
day_thresh = 2

snotel_sdd, _ = proc.calc_sdd(snotel_df[f'{sitename} Snow Depth (cm) End of Day Values']/100, 
                              snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
print(snotel_sdd)
# Convert model data to Pandas Series
classic_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(snow_var_data.data), index=snow_var_data.time.values),
                              snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
print(classic_sdd)

# Convert model data to Pandas Series
modis_hrrr_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(snow_updated_var_data.data), index=snow_updated_var_data.time.values),
                              snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
print(modis_hrrr_sdd)

In [ ]:
%%time
fig, ax = plt.subplots(1, figsize=(8, 4))
snow_var_data.plot(ax=ax, color='gray', label='iSnobal-HRRR', linewidth=1)
snow_updated_var_data.plot(ax=ax, color=isnobalcolors[1], label='MODIS-HRRR')

# Plot WY time series of snow depth
(snotel_df[f'{sitename} Snow Depth (cm) End of Day Values']/100).plot(ax=ax, 
                                                                label=f'{sitename} Snow Depth [m]', 
                                                                linestyle=linestyle, 
                                                                linewidth=linewidth, 
                                                                # color=snotelcolors[0], 
                                                                marker=marker)

plt.legend(bbox_to_anchor=(1,1))
plt.title('Snow Depth')
plt.xlabel('')

# Add callouts of disappearance dates with vertical lines
plt.axvline(snotel_sdd, color=snotelcolors[0], linestyle='--', linewidth=0.75, label='SNOTEL SDD')
plt.axvline(classic_sdd, color='gray', linestyle='--', linewidth=0.75, label='iSnobal-HRRR SDD')
plt.axvline(modis_hrrr_sdd, color='coral', linestyle='--', linewidth=0.75, label='MODIS-HRRR SDD')

# Add arrows pointing left towards the vertical lines above, in the same color as the line
plt.annotate(f'{snotel_sdd.strftime("%Y-%m-%d")}', xy=(snotel_sdd, 1), xytext=(snotel_sdd, 1.2),
             color=snotelcolors[0], arrowprops=dict(color=snotelcolors[0], arrowstyle='->'))
plt.annotate(f'{classic_sdd.strftime("%Y-%m-%d")}', xy=(classic_sdd, 0.5), xytext=(classic_sdd, 0.6),
             color='gray', arrowprops=dict(color='gray', arrowstyle='->'))
plt.annotate(f'{modis_hrrr_sdd.strftime("%Y-%m-%d")}', xy=(modis_hrrr_sdd, 0.5), xytext=(modis_hrrr_sdd, 0.75),
             color='coral', arrowprops=dict(color='coral', arrowstyle='->'))

# Annotate with dates of disappearance dates abbreviating to YYYMMDD

# plt.text(snotel_sdd, 1.5, f'{snotel_sdd.strftime("%Y-%m-%d")}', horizontalalignment='left', color=snotelcolors[0])
# plt.text(classic_sdd, 0.5, f'{classic_sdd.strftime("%Y-%m-%d")}', horizontalalignment='left', color='gray')
# plt.text(modis_hrrr_sdd, 2.5, f'{modis_hrrr_sdd.strftime("%Y-%m-%d")}', horizontalalignment='left', color='coral')

In [ ]:
for jdx in range(len(sitenums)):
    try:
        snotel_df, gdf, sitenum, sitename = proc.get_snotel_df_pt(snotel_dir=snotel_dir, sitenums=sitenums, WY=WY, jdx=jdx)

        snow_var_data = [ds[thisvar].sel(x=list(gdf.geometry.x.values), y=list(gdf.geometry.y.values), method='nearest') for ds in ds_list]
        snow_updated_var_data = [ds[thisvar].sel(x=list(gdf.geometry.x.values), y=list(gdf.geometry.y.values), method='nearest') for ds in ds_sol_list]

        # Concatenate all the days
        snow_var_data = xr.concat(snow_var_data, dim='time')
        snow_updated_var_data = xr.concat(snow_updated_var_data, dim='time')

        # Get disappearance dates
        snow_name = 'Snow Depth'
        verbose = False
        day_thresh = 2

        snotel_sdd, _ = proc.calc_sdd(snotel_df[f'{sitename} Snow Depth (cm) End of Day Values']/100, 
                                    snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
        print(snotel_sdd)
        # Convert model data to Pandas Series
        classic_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(snow_var_data.data), index=snow_var_data.time.values),
                                    snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
        print(classic_sdd)

        # Convert model data to Pandas Series
        modis_hrrr_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(snow_updated_var_data.data), index=snow_updated_var_data.time.values),
                                    snow_name=snow_name, verbose=verbose, day_thresh=day_thresh)
        print(modis_hrrr_sdd)

        figsize = (8, 4)
        fig, ax = plt.subplots(1, figsize=figsize)
        snow_var_data.plot(ax=ax, color='gray', label='iSnobal-HRRR', linewidth=1)
        snow_updated_var_data.plot(ax=ax, color=isnobalcolors[1], label='MODIS-HRRR')

        # Plot WY time series of snow depth
        (snotel_df[f'{sitename} Snow Depth (cm) End of Day Values']/100).plot(ax=ax, 
                                                                        label=f'{sitename} Snow Depth [m]', 
                                                                        linestyle=linestyle, 
                                                                        linewidth=1,
                                                                        color=snotelcolors[0], 
                                                                        marker=marker)

        # plt.legend(bbox_to_anchor=(1,-0.2), ncol=3, alignment='center')
        plt.legend(bbox_to_anchor=(1,1), alignment='center')
        plt.title('Snow Depth')
        plt.xlabel('')

        # Add callouts of disappearance dates with vertical lines
        plt.axvline(snotel_sdd, color=snotelcolors[0], linestyle='--', linewidth=0.75, label='SNOTEL SDD')
        plt.axvline(classic_sdd, color='gray', linestyle='--', linewidth=0.75, label='iSnobal-HRRR SDD')
        plt.axvline(modis_hrrr_sdd, color='coral', linestyle='--', linewidth=0.75, label='MODIS-HRRR SDD')



        # Add arrows pointing left towards the vertical lines above, in the same color as the line
        # get fraction of max y value for the y position of the arrow
        ymax = ax.get_ylim()[1]

        plt.annotate(f'{snotel_sdd.strftime("%Y-%m-%d")}', xy=(snotel_sdd, ymax*0.2), xytext=(snotel_sdd, ymax*0.3),
                    color=snotelcolors[0], arrowprops=dict(color=snotelcolors[0], arrowstyle='->'))
        plt.annotate(f'{classic_sdd.strftime("%Y-%m-%d")}', xy=(classic_sdd, ymax*0.8), xytext=(classic_sdd, ymax*0.9),
                    color='gray', arrowprops=dict(color='gray', arrowstyle='->'))
        plt.annotate(f'{modis_hrrr_sdd.strftime("%Y-%m-%d")}', xy=(modis_hrrr_sdd, ymax*0.6), xytext=(modis_hrrr_sdd, ymax*0.7),
                    color='coral', arrowprops=dict(color='coral', arrowstyle='->'))
    except:
        print('hit a snag')